# 20. Speech AI — Call Center Analytics

## Banking Business Case

Call center conversations contain valuable information about:

- customer complaints,
- product needs,
- fraud reports,
- service quality,
- agent performance,
- customer sentiment,
- recurring operational problems.

Historically, banks may analyze only a small sample of calls manually.

**Speech AI** can turn conversations into structured analytics:

```text
Voice Call
   ↓
Speech-to-Text (ASR)
   ↓
Transcript
   ↓
Speaker Diarization
   ↓
Topic / Intent Detection
   ↓
Sentiment Analysis
   ↓
Entity / Keyword Extraction
   ↓
Conversation Analytics
   ↓
Agent & Customer Insights
   ↓
Action / Monitoring
```

> This notebook uses synthetic call metadata and transcript snippets. It does not process real customer recordings.

## 1. What is Speech AI?

Speech AI combines:

```text
Automatic Speech Recognition (ASR)
+
Speaker Diarization
+
NLP
+
Sentiment Analysis
+
Topic / Intent Classification
+
Conversation Analytics
```

The goal is not merely:

> "Convert speech to text."

The goal is:

> **Convert conversations into actionable business intelligence.**

## 2. Example Banking Call

Customer:

> "Saya sudah menunggu lama dan transaksi saya belum selesai."

Speech AI can produce:

```text
Intent     = Transaction Issue
Sentiment  = Negative
Topic      = Transaction
Urgency    = High
Resolution = Pending
```

This can trigger:

```text
Agent Coaching
+
Customer Follow-up
+
Operational Alert
```


## 3. End-to-End Architecture

```text
             Customer Call
                   ↓
             Audio Recording
                   ↓
             Audio Quality
                   ↓
          Speech-to-Text (ASR)
                   ↓
         Speaker Diarization
                   ↓
              Transcript
                   ↓
       ┌───────────┼───────────┐
       ↓           ↓           ↓
    Intent     Sentiment    Keywords
       ↓           ↓           ↓
       └───────────┼───────────┘
                   ↓
          Conversation Analytics
                   ↓
       ┌───────────┼────────────┐
       ↓           ↓            ↓
 Customer       Agent       Operations
 Insight      Coaching       Insight
       └───────────┼────────────┘
                   ↓
              Monitoring
```


## 4. Project Objectives

This project answers:

1. What is the customer calling about?
2. Is the customer satisfied or frustrated?
3. What products/issues are discussed?
4. Is the issue resolved?
5. How much time is spent on the call?
6. How often is the customer transferred?
7. Which calls need follow-up?
8. Which agents need coaching?
9. What recurring problems exist?
10. Which calls may require escalation?

## 5. Import Libraries

This step imports the libraries used throughout the notebook — pandas and NumPy for data handling, re for transcript text processing, and Matplotlib/Seaborn for visualisation.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",100)

calls = pd.read_csv(
    "call_center_analytics_features.csv",
    parse_dates=["Call_Date"]
)

taxonomy = pd.read_csv("speech_topic_taxonomy.csv")

print("Calls:", len(calls))
display(calls.head())
display(taxonomy)
# --- Save all outputs to the local output/ folder ---
from pathlib import Path

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)


def save_plot(filename, dpi=150):
    """Save the current matplotlib figure into the output/ folder."""
    path = OUTPUT_DIR / filename
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"Saved: {path}")


## 6. Dataset Dictionary

Call records are documented — transcript, detected intent, sentiment, handling time and resolution outcome — defining the measurements speech analytics produces per call.


In [ ]:
dictionary = pd.DataFrame({
    "Field": [
        "Call_ID","Customer_ID","Agent_ID","Call_Date","Channel",
        "Product","Intent","Sentiment","Duration_Sec","Hold_Sec",
        "Transfer_Count","Resolution","CSAT","ASR_Confidence",
        "Silence_Ratio","Agent_Talk_Ratio","Customer_Talk_Ratio",
        "FCR","QA_Score"
    ],
    "Description": [
        "Unique call identifier","Synthetic customer ID","Agent identifier",
        "Call date","Inbound / outbound","Banking product",
        "Call reason","Customer sentiment","Call duration",
        "Time customer was placed on hold","Number of transfers",
        "Call outcome","Customer satisfaction score",
        "Speech recognition confidence","Share of silence",
        "Share of agent speaking time","Share of customer speaking time",
        "First Contact Resolution indicator","Synthetic quality score"
    ]
})
display(dictionary)

## 7. Data Quality

Missing transcripts and inconsistent labels are checked. Unusable transcripts silently reduce the coverage of the analytics, so the effective sample is quantified here.


In [ ]:
print("Missing values:")
display(calls.isna().sum().to_frame("Missing"))

print("Calls by channel:")
display(calls["Channel"].value_counts().to_frame("Calls"))

print("Calls by intent:")
display(calls["Intent"].value_counts().to_frame("Calls"))

## 8. Speech-to-Text (ASR)

Automatic Speech Recognition converts:

```text
Audio
  ↓
Text Transcript
```

Examples of production technologies include:

- cloud speech APIs,
- open-source speech models,
- enterprise speech platforms.

Important ASR metrics:

```text
Word Error Rate (WER)
Character Error Rate (CER)
ASR Confidence
```

For Indonesian banking calls, the model should handle:

- Bahasa Indonesia,
- English terms,
- product names,
- abbreviations,
- numbers,
- financial terminology,
- code-switching.

## 9. Speaker Diarization

Diarization answers:

> Who spoke when?

Example:

```text
[00:01] CUSTOMER: Saya ingin menanyakan kartu saya.
[00:05] AGENT: Baik, saya bantu cek.
[00:12] CUSTOMER: Karena transaksi saya gagal.
```

This enables:

- talk-time analysis,
- interruption analysis,
- customer vs agent sentiment,
- agent coaching,
- conversation structure analysis.

## 10. Transcript Analytics

A transcript can be transformed into:

```text
Raw Speech
   ↓
Transcript
   ↓
Tokens / Sentences
   ↓
Intent
Sentiment
Entities
Topics
Keywords
Actions
```

For this notebook, synthetic transcript snippets are supplied.

In [ ]:
display(
    calls[
        ["Call_ID","Intent","Sentiment","Transcript_Snippet"]
    ].head(15)
)

## 11. Intent Classification

Intent answers:

> **Why is the customer calling?**

Examples:

```text
Complaint
Information Request
Transaction Issue
Fraud Report
Product Inquiry
Cancellation
Payment Issue
```

A production ML classifier could use:

```text
TF-IDF + Logistic Regression
BERT / Transformer
Indonesian language model
LLM classification
```

The target variable is the call intent.

## 12. Intent Distribution

The share of calls per detected intent — balance inquiry, card block, complaint and so on — shows what customers actually call about and where self-service could deflect the most volume.


In [ ]:
plt.figure(figsize=(9,5))
sns.countplot(
    data=calls,
    y="Intent",
    order=calls["Intent"].value_counts().index
)
plt.title("Call Center Intent Distribution")
plt.xlabel("Calls")
plt.ylabel("Intent")
save_plot("12-intent-distribution.png")
plt.show()

## 13. Sentiment Analysis

Sentiment estimates the emotional orientation of the conversation:

```text
Positive
Neutral
Negative
```

Example:

```text
"I am very satisfied with the solution."
→ Positive

"I want to know the status."
→ Neutral

"I have called several times and nobody solved it."
→ Negative
```

Production systems may use:

- sentiment classifiers,
- transformer models,
- LLMs,
- acoustic + linguistic signals.

## 14. Sentiment Distribution

The balance of positive, neutral and negative calls is visualised, and the negative share per intent highlights which customer journeys frustrate callers most and deserve first attention.


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(
    data=calls,
    x="Sentiment",
    order=["Positive","Neutral","Negative"]
)
plt.title("Customer Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Calls")
save_plot("14-sentiment-distribution.png")
plt.show()

## 15. Sentiment Score

A continuous sentiment score is often useful:

```text
0.00 → Very Negative
0.50 → Neutral
1.00 → Very Positive
```

It allows:

- trend analysis,
- threshold alerts,
- customer-level monitoring,
- correlation with CSAT.

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(calls["Sentiment_Score"],bins=30,kde=True)
plt.title("Sentiment Score Distribution")
plt.xlabel("Sentiment Score")
save_plot("15-sentiment-score.png")
plt.show()

## 16. Topic / Keyword Extraction

Speech AI can detect topics such as:

```text
Fraud
KPR
Credit Card
Payment
Mobile Banking
Cancellation
```

Methods:

- keyword rules,
- TF-IDF,
- topic modeling,
- NER,
- embeddings,
- LLM-based extraction.

In [ ]:
topic_keywords = {
    "Fraud": ["tidak melakukan","mencurigakan","bukan saya"],
    "Complaint": ["kecewa","menunggu lama","belum selesai"],
    "Payment": ["pembayaran","cicilan","tagihan"],
    "Card": ["kartu","kartu kredit","debit"],
    "Mobile": ["mobile banking","aplikasi","login"]
}

def detect_topics(text):
    text = str(text).lower()
    found=[]
    for topic, kws in topic_keywords.items():
        if any(k in text for k in kws):
            found.append(topic)
    return ", ".join(found) if found else "Other"

calls["Detected_Topics"] = calls["Transcript_Snippet"].apply(detect_topics)

display(calls[["Call_ID","Transcript_Snippet","Detected_Topics"]].head(20))

## 17. Fraud / Security Escalation

Speech analytics can identify phrases such as:

```text
"bukan saya"
"tidak melakukan transaksi"
"transaksi mencurigakan"
```

These signals do **not automatically prove fraud**.

Instead, they can create an alert:

```text
Speech Signal
     ↓
Potential Fraud Indicator
     ↓
Case Management
     ↓
Fraud Investigation
```

This is an important distinction between **signal detection** and **fraud determination**.

In [ ]:
calls["Fraud_Signal"] = (
    calls["Intent"].eq("Fraud Report") |
    calls["Detected_Topics"].str.contains("Fraud", na=False)
)

display(
    calls["Fraud_Signal"].value_counts()
    .rename_axis("Fraud Signal")
    .to_frame("Calls")
)

## 18. Call Duration Analytics

Important metrics:

```text
AHT = Average Handle Time

AHT =
Total Handling Time
-------------------
Number of Calls
```

Long calls are not necessarily bad.

A long call may indicate:

- complex customer issue,
- fraud investigation,
- technical problem,
- multiple products,
- repeated explanations.

In [ ]:
aht = calls["Duration_Sec"].mean()

print("Average Handle Time:", round(aht/60,2), "minutes")

plt.figure(figsize=(9,5))
sns.histplot(calls["Duration_Sec"]/60,bins=35,kde=True)
plt.title("Call Duration Distribution")
plt.xlabel("Minutes")
save_plot("18-call-duration-analytics.png")
plt.show()

## 19. Hold Time

High hold time may indicate:

- slow system lookup,
- knowledge gaps,
- complex approval,
- transfer to another team.

Track:

```text
Average Hold Time
Hold Ratio
Calls with Hold
```

In [ ]:
calls["Hold_Ratio"] = (
    calls["Hold_Sec"] /
    calls["Duration_Sec"]
).clip(0,1)

print("Average hold time:",
      round(calls["Hold_Sec"].mean(),1),"seconds")

print("Average hold ratio:",
      round(calls["Hold_Ratio"].mean()*100,2),"%")

## 20. Transfer Analytics

Transfers can indicate:

```text
Customer → Agent A
             ↓
           Agent B
             ↓
           Back Office
```

Repeated transfers can increase customer effort.

Useful metrics:

- average transfers,
- transfer rate,
- transfer rate by intent,
- transfer rate by agent.

In [ ]:
transfer_rate = (calls["Transfer_Count"] > 0).mean()
print("Transfer Rate:", round(transfer_rate*100,2), "%")

transfer_by_intent = (
    calls.groupby("Intent")["Transfer_Count"]
    .mean()
    .sort_values(ascending=False)
)
display(transfer_by_intent.to_frame("Avg Transfers"))

## 21. First Contact Resolution (FCR)

FCR measures whether the issue was resolved without requiring additional transfers or follow-up.

Simplified synthetic definition:

```text
Resolved
AND
Zero Transfers
```

FCR is a major call-center operational KPI.

In [ ]:
fcr_rate = calls["FCR"].mean()
print("FCR Rate:", round(fcr_rate*100,2), "%")

fcr_by_intent = (
    calls.groupby("Intent")["FCR"]
    .mean()
    .sort_values()
)
display((fcr_by_intent*100).round(2).to_frame("FCR %"))

## 22. Customer Satisfaction (CSAT)

CSAT is measured here from 1–5.

Analyze:

```text
CSAT vs Sentiment
CSAT vs FCR
CSAT vs Duration
CSAT vs Hold Time
CSAT vs Transfer Count
```

The goal is to discover which conversation characteristics are associated with customer experience.

In [ ]:
plt.figure(figsize=(9,5))
sns.boxplot(data=calls, x="Sentiment", y="CSAT", order=["Positive","Neutral","Negative"])
plt.title("CSAT by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("CSAT")
save_plot("22-customer-satisfaction-csat.png")
plt.show()

## 23. Agent Performance Analytics

Do not evaluate agents using a single metric.

Combine:

```text
CSAT
FCR
AHT
Transfer Rate
Hold Time
QA Score
Sentiment
```

A long AHT alone does not necessarily indicate poor performance.

Context matters.

In [ ]:
agent_perf = (
    calls.groupby("Agent_ID")
    .agg(
        Calls=("Call_ID","count"),
        Avg_AHT_Sec=("Duration_Sec","mean"),
        FCR=("FCR","mean"),
        Avg_CSAT=("CSAT","mean"),
        Avg_Hold_Sec=("Hold_Sec","mean"),
        Transfer_Rate=("Transfer_Count",lambda x:(x>0).mean()),
        Avg_QA=("QA_Score","mean")
    )
    .reset_index()
)

display(agent_perf.head(15))

## 24. Agent QA Score

Synthetic QA score combines:

```text
FCR
Transfer behavior
ASR quality
CSAT
```

In a production quality-assurance system, QA may also include:

- greeting compliance,
- identity verification,
- policy adherence,
- mandatory disclosure,
- empathy,
- resolution quality,
- prohibited statements.

In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(agent_perf["Avg_QA"],bins=25,kde=True)
plt.title("Agent QA Score Distribution")
plt.xlabel("Average QA Score")
save_plot("24-agent-qa-score.png")
plt.show()

## 25. Call Center Dashboard Metrics

Recommended executive dashboard:

```text
Total Calls
AHT
FCR
CSAT
Negative Sentiment %
Transfer Rate
Fraud Signals
Human Review / Escalation
Top Intents
Top Complaints
```

Drill-down:

```text
Bank
 ↓
Region
 ↓
Team
 ↓
Agent
 ↓
Call
 ↓
Transcript
```


In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total Calls","AHT (min)","FCR %","CSAT",
        "Negative Sentiment %","Transfer Rate %",
        "Fraud Signal %"
    ],
    "Value": [
        len(calls),
        round(calls["Duration_Sec"].mean()/60,2),
        round(calls["FCR"].mean()*100,2),
        round(calls["CSAT"].mean(),2),
        round((calls["Sentiment"]=="Negative").mean()*100,2),
        round((calls["Transfer_Count"]>0).mean()*100,2),
        round(calls["Fraud_Signal"].mean()*100,2)
    ]
})
display(summary)

## 26. Conversation Funnel

A useful operational funnel:

```text
All Calls
   ↓
Issue Identified
   ↓
Intent Classified
   ↓
Solution Provided
   ↓
Resolved
   ↓
Customer Satisfied
```

Each drop-off can be investigated.

## 27. Alerting

Potential real-time or near-real-time alerts:

### Customer Experience

```text
Strong negative sentiment
+
Repeated complaint
→ Supervisor alert
```

### Fraud

```text
Fraud-related phrase
→ Security workflow
```

### Operations

```text
Repeated same complaint
→ Product / IT investigation
```

### Agent Coaching

```text
Low QA
+
High transfer
+
Low FCR
→ Coaching review
```


## 28. Advanced AI Architecture

A mature Speech AI platform may use:

```text
Audio
 ↓
ASR
 ↓
Diarization
 ↓
Speaker-separated Transcript
 ↓
Embedding Model
 ↓
Intent Classifier
 ↓
Sentiment Model
 ↓
NER / Entity Extraction
 ↓
LLM Summarization
 ↓
Quality & Compliance Models
 ↓
Event / Alert Engine
 ↓
Data Warehouse
```

This can support both analytics and operational workflows.

## 29. Automatic Call Summary

A production LLM can generate:

```text
Customer Issue:
Transaction was not recognized by customer.

Action Taken:
Agent verified transaction details and initiated investigation.

Next Action:
Fraud team follow-up required.

Customer Sentiment:
Negative.

Priority:
High.
```

The summary should be grounded in the transcript and subject to appropriate controls.

## 30. Compliance & Responsible AI

Call-center speech data can contain sensitive information.

Controls should include:

- consent / legal basis where applicable,
- encryption,
- role-based access,
- transcript masking,
- PII redaction,
- retention policies,
- audit logging,
- model monitoring,
- human review for consequential decisions.

Do not use sentiment or speech analytics as an unsupported proxy for sensitive personal characteristics.

## 31. Production Data Model

Recommended tables:

```text
CALL
 ├── Call_ID
 ├── Customer_ID
 ├── Agent_ID
 ├── Start_Time
 ├── Duration
 └── Channel

TRANSCRIPT
 ├── Call_ID
 ├── Speaker
 ├── Timestamp
 └── Text

CALL_INTENT
 ├── Call_ID
 ├── Intent
 └── Confidence

CALL_SENTIMENT
 ├── Call_ID
 ├── Speaker
 ├── Sentiment
 └── Confidence

CALL_ENTITY
 ├── Call_ID
 ├── Entity_Type
 └── Entity_Value

CALL_QA
 ├── Call_ID
 ├── QA_Score
 └── Rule_Result
```

## 32. Common Mistakes

1. Treating transcript as perfect.
2. Ignoring ASR errors.
3. Ignoring speaker diarization.
4. Using sentiment as an absolute truth.
5. Evaluating agents using one KPI.
6. Automatically labeling fraud from keywords.
7. Ignoring PII.
8. No human review.
9. No model confidence.
10. No drift monitoring.
11. No audit trail.
12. Optimizing AHT at the expense of resolution quality.

## 33. Final Executive Summary

### Business Question

> **What can the bank learn from customer conversations at scale?**

### End-to-End

```text
Voice Call
    ↓
ASR
    ↓
Transcript
    ↓
Speaker Diarization
    ↓
Intent Detection
    ↓
Sentiment Analysis
    ↓
Topic / Entity Extraction
    ↓
Conversation Analytics
    ↓
Agent Analytics
    ↓
Customer Analytics
    ↓
Operational Alerts
    ↓
Monitoring
```

### Main Banking Applications

- Call Center Analytics
- Customer Experience
- Agent Coaching
- Complaint Analytics
- Fraud Signals
- Quality Assurance
- Compliance Monitoring
- Product Feedback
- Root Cause Analysis
- Customer Voice / VoC

**Key idea:**

> Speech AI turns unstructured conversations into structured signals that can support customer experience, operations, risk, and service improvement.

## 34. Save Outputs

All key result tables and figures are saved into the local `output/` folder, so results survive after the kernel is restarted and can be shared without re-running the notebook.


In [ ]:
agent_perf.to_csv(OUTPUT_DIR / "agent_performance.csv", index=False)
print("Saved:", OUTPUT_DIR / "agent_performance.csv")
summary.to_csv(OUTPUT_DIR / "call_center_dashboard_metrics.csv", index=False)
print("Saved:", OUTPUT_DIR / "call_center_dashboard_metrics.csv")


# List everything that was written to output/
for _p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", _p)
